# 6. Rejeu H1 : que valent les prédictions, match après match ?

**Question** : sur tous les matchs du top 5 rejoués depuis 2021-22, que valent les prédictions du modèle M3 (H1) face à la référence B1, par championnat, par saison, au fil de la saison, et en fonction des variables que le modèle a lues ?

> Notebook **descriptif**. Il lit `data/inference/jeu_rejeu_h1.csv`, un match par ligne (variables lues, prédictions finales, score réel, mesures), produit par `python -m foot_predictor.inference jeu-rejeu` et complété à chaque passage planifié (ADR-0054).
> Il ne choisit aucune variable ni aucun réglage, et ne change aucun modèle.
>
> **Précautions de lecture**
> - Les saisons **2021-22 à 2024-25** ont servi à choisir les modèles et les réglages : seules **2025-26 et 2026-27** sont vierges (colonne `period`). Elles restent toujours séparées dans les tableaux « par période ».
> - Les analyses sont nombreuses : un intervalle qui exclut 0 dans l'une d'elles ne prouve rien seul (environ 5 % le font par hasard).
> - Le « bon vainqueur » (`hit_1x2`) est déduit des deux λ par des Poissons indépendants : le protocole ne l'a pas validé.
> - Les intervalles (95 %) viennent d'un rééchantillonnage de blocs de journées, avec la méthode de `rejeu-stats` mais **`RESAMPLES` tirages** au lieu des 10 000 du protocole : ils bougent un peu d'un lancement à l'autre.

In [ ]:
import json
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

# Mêmes fonctions que `rejeu-stats` : le notebook retrouve donc les tableaux de reports/inference/rejeu_stats_*.md.
from foot_predictor.inference import rejeu_stats as rs

pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 40)
RESAMPLES = 1000  # le protocole en demande 10 000 (ADR-0037) ; 1 000 suffisent pour lire


def repo_root() -> Path:
    here = Path.cwd().resolve()
    for folder in (here, *here.parents):
        if (folder / "pyproject.toml").exists():
            return folder
    raise FileNotFoundError("pyproject.toml introuvable : lancer le notebook depuis le dépôt")


FILE = repo_root() / "data" / "inference" / "jeu_rejeu_h1.csv"
jeu = pd.read_csv(FILE, parse_dates=["match_day"])
order = [rs.tranche_label(low) for low, _ in rs.TRANCHES]
jeu["tranche"] = pd.Categorical(jeu["tranche"], categories=order, ordered=True)
meta = json.loads(FILE.with_suffix(".meta.json").read_text(encoding="utf-8"))
print(f"{len(jeu)} matchs du {meta['first_day']} au {meta['last_day']} ; mis à jour le {meta['updated_at']}")
print("Version des données :", meta["data_version"])
for season, versions in meta["models"].items():
    print(" ", season, "->", ", ".join(versions))

## Contrôle : combien de matchs par championnat et par saison

La saison 2026-27 est en cours : elle grossit à chaque passage. Une saison n'est dite **terminée** (`season_complete`) quand ses journées jouées atteignent 90 % de son nombre de journées.

In [ ]:
display(jeu.groupby(["competition", "season"]).size().unstack())
display(jeu.groupby("period").size().to_frame("matchs"))
display(
    jeu.groupby(["competition", "season"])["season_complete"]
    .first()
    .unstack()
    .replace({True: "terminée", False: "en cours"})
)

## Les colonnes

Une ligne = un match. Les variables lues par le modèle sont `home_*` (ligne qui prédit le λ du domicile) et `away_*` (ligne du λ de l'extérieur) ; `opp_*` est l'adversaire de la ligne, donc `home_opp_elo_pre` égale `away_elo_pre`.

Chaque modèle de saison lit 11 variables par côté. La demi-vie des moyennes glissantes (`_h120` ou `_h240`) est choisie par pli : 120 jours pour 2022-23, 240 pour les autres saisons. **Une colonne `_h120` est donc vide hors de 2022-23, et une colonne `_h240` vide en 2022-23** : ce n'est pas une donnée manquante, le modèle de la saison ne la lit pas.

In [ ]:
variables = [
    c
    for c in jeu.columns
    if c.startswith(("home_", "away_")) and c not in ("home_team", "away_team", "home_goals", "away_goals")
]
print(len(variables), "colonnes de variables :", ", ".join(variables))
display(jeu.drop(columns=variables).head(3).T)

## Outil de lecture : un tableau à plat

`rs.table` rend, pour chaque groupe, les mesures et l'écart à B1 avec son intervalle. L'**écart à B1** est `log-loss(B1) − log-loss(modèle)` : **positif = le modèle fait mieux** que la simple moyenne de buts du championnat.

In [ ]:
def flat(rows: list[dict]) -> pd.DataFrame:
    out = []
    for row in rows:
        b1 = row["b1"] or {}
        delta = b1.get("delta") or {}
        out.append(
            {
                **row["group"],
                "matchs": row["n"],
                "log_loss": row["log_loss"],
                "rps": row["rps"],
                "brier_>2,5": row["brier_over_2_5"],
                "bons_>2,5": row["hit_over_2_5"],
                "buts_prévus": row["predicted_total"],
                "buts_observés": row["observed_total"],
                "bon_vainqueur": row["hit_1x2"],
                "écart_B1": delta.get("mean"),
                "ic_bas": delta.get("low"),
                "ic_haut": delta.get("high"),
            }
        )
    return pd.DataFrame(out)


FORMAT = {c: "{:.4f}" for c in ("log_loss", "rps", "brier_>2,5", "écart_B1", "ic_bas", "ic_haut")}
FORMAT |= {"bons_>2,5": "{:.1%}", "bon_vainqueur": "{:.1%}", "buts_prévus": "{:.2f}", "buts_observés": "{:.2f}"}


def show(frame: pd.DataFrame) -> None:
    display(frame.style.format(FORMAT, na_rep="—").hide(axis="index"))

## Vue d'ensemble, par période

Validation (2021-22 à 2024-25, qui a servi aux choix) puis test scellé (2025-26 et 2026-27, vierges), puis tout.

In [ ]:
show(flat(rs.period_rows(jeu, RESAMPLES)))

**Repères naïfs** pour situer les taux de bons : toujours « plus de 2,5 » ou « moins de 2,5 » (le côté majoritaire de chaque championnat), et toujours le domicile.

In [ ]:
naive = pd.DataFrame(
    {
        "plus_de_2,5_observé": jeu.groupby("competition")["over"].mean(),
        "côté_majoritaire": jeu.groupby("competition")["over"].mean().map(lambda p: max(p, 1 - p)),
        "modèle": jeu.groupby("competition")["hit_over"].mean(),
    }
)
naive["domicile_gagne"] = (jeu["home_goals"] > jeu["away_goals"]).groupby(jeu["competition"]).mean()
naive["vainqueur_modèle"] = jeu.groupby("competition")["hit_1x2"].mean()
display(naive.style.format("{:.1%}"))

## Par championnat (toutes saisons)

In [ ]:
show(flat(rs.table(jeu, ["competition"], RESAMPLES)))

## Par saison

In [ ]:
show(flat(rs.table(jeu, ["season"], RESAMPLES)))

## Par saison et par championnat

Environ 300 à 380 matchs par case : les intervalles sont larges, et beaucoup incluent 0. La carte donne l'écart à B1 (vert : le modèle fait mieux).

In [ ]:
fine = flat(rs.table(jeu, ["competition", "season"], RESAMPLES))
grid = fine.pivot(index="competition", columns="season", values="écart_B1")
sure = fine.assign(sûr=(fine["ic_bas"] > 0) | (fine["ic_haut"] < 0)).pivot(
    index="competition", columns="season", values="sûr"
)
fig, ax = plt.subplots(figsize=(8, 3.5))
limit = np.nanmax(np.abs(grid.to_numpy()))
image = ax.imshow(grid.to_numpy(), cmap="RdYlGn", vmin=-limit, vmax=limit, aspect="auto")
ax.set_xticks(range(grid.shape[1]), grid.columns)
ax.set_yticks(range(grid.shape[0]), grid.index)
for i in range(grid.shape[0]):
    for j in range(grid.shape[1]):
        mark = "*" if sure.iloc[i, j] else ""
        ax.text(j, i, f"{grid.iloc[i, j]:+.3f}{mark}", ha="center", va="center", fontsize=9)
ax.set_title("Écart à B1 (log-loss) ; * : intervalle à 95 % qui exclut 0")
fig.colorbar(image, ax=ax)
plt.tight_layout()
plt.show()
show(fine)

## Plus on avance dans la saison, plus c'est fiable ?

Seules les **saisons terminées** entrent ici : une saison en cours n'a que ses premières journées, ce qui fausserait les tranches et donnerait des pentes extrapolées. L'écart à B1 est la bonne mesure pour comparer des périodes (le log-loss absolu dépend aussi du nombre de buts : on en marque plus en début de saison).

In [ ]:
done = jeu[jeu["season_complete"] & jeu["tranche"].notna()]
print("Saisons terminées :", ", ".join(sorted(done["season"].unique())), f"({len(done)} matchs)")
tranches = flat(rs.table(done, ["tranche"], RESAMPLES))
show(tranches)

fig, axes = plt.subplots(1, 2, figsize=(10, 3.5))
x = np.arange(len(tranches))
axes[0].plot(x, tranches["log_loss"], marker="o")
axes[0].set_title("Log-loss par tranche de journées")
axes[1].errorbar(
    x,
    tranches["écart_B1"],
    yerr=[tranches["écart_B1"] - tranches["ic_bas"], tranches["ic_haut"] - tranches["écart_B1"]],
    fmt="o-",
    capsize=4,
)
axes[1].axhline(0, color="grey", lw=0.8)
axes[1].set_title("Écart à B1 par tranche (IC 95 %)")
for ax in axes:
    ax.set_xticks(x, tranches["tranche"])
    ax.set_xlabel("journées")
plt.tight_layout()
plt.show()

**Pente de l'erreur selon l'avancement** (de la 1re à la dernière journée, droite des moindres carrés, après retrait des moyennes du groupe). Une pente de log-loss **négative** ou une pente d'écart à B1 **positive** voudrait dire « meilleur en fin de saison ». La ligne `tous` est la plus informative ; les lignes `saison` (25 groupes) sont bruitées, et sur 25 intervalles à 95 %, environ un exclut 0 par hasard.

In [ ]:
names = jeu.groupby("api_league_id")["competition"].first().to_dict()
rows = rs.slopes(done, names, RESAMPLES)


def pente(found):
    return "—" if found is None else f"{found['slope']:+.4f} [{found['low']:+.4f} ; {found['high']:+.4f}]"


table = pd.DataFrame(
    [
        {
            **r["group"],
            "matchs": r["n"],
            "log_loss": pente(r["log_loss"]),
            "rps": pente(r["rps"]),
            "écart_B1": pente(r["delta_b1"]),
        }
        for r in rows
    ]
)
display(table[table["kind"] != "saison"].fillna("").style.hide(axis="index"))
print(rs.slope_summary(rows))

## Calibration de « plus de 2,5 buts »

Pour chaque tranche de P(T > 2,5) prédite, la fréquence observée. Une prédiction bien calibrée suit la diagonale.

In [ ]:
bins = pd.cut(jeu["p_over"], [0, 0.35, 0.4, 0.45, 0.5, 0.55, 0.6, 0.65, 0.7, 1.0])
cal = jeu.groupby(bins, observed=True).agg(matchs=("over", "size"), prédit=("p_over", "mean"), observé=("over", "mean"))
display(cal.style.format({"prédit": "{:.3f}", "observé": "{:.3f}"}))
fig, ax = plt.subplots(figsize=(4.5, 4.5))
ax.plot([0.3, 0.8], [0.3, 0.8], color="grey", lw=0.8)
ax.scatter(cal["prédit"], cal["observé"], s=cal["matchs"] / 8)
ax.set_xlabel("P(T > 2,5) prédite")
ax.set_ylabel("fréquence observée")
plt.tight_layout()
plt.show()

## Ce que permettent les variables : buts prévus et observés selon la force relative

C'est ce que le rapport ne donne pas. Les matchs sont classés par l'écart d'Elo avant le match (domicile moins extérieur) en cinq groupes de même taille ; on compare les buts prévus (λ) aux buts marqués.

In [ ]:
jeu["écart_elo"] = jeu["home_elo_pre"] - jeu["away_elo_pre"]
groupe = pd.qcut(jeu["écart_elo"], 5)
strength = jeu.groupby(groupe, observed=True).agg(
    matchs=("écart_elo", "size"),
    λ_domicile=("lambda_home", "mean"),
    buts_domicile=("home_goals", "mean"),
    λ_extérieur=("lambda_away", "mean"),
    buts_extérieur=("away_goals", "mean"),
    écart_B1=("delta_b1", "mean"),
)
display(strength.style.format("{:.3f}", subset=strength.columns[1:]))

## Les matchs les plus mal prédits

Ceux dont la perte (log-loss du total) est la plus forte : à regarder avant de conclure à un défaut du modèle, un score très rare reste très rare.

In [ ]:
cols = [
    "match_day",
    "competition",
    "home_team",
    "away_team",
    "home_goals",
    "away_goals",
    "expected_total",
    "p_over",
    "log_loss",
]
display(
    jeu.nlargest(10, "log_loss")[cols]
    .style.format({"expected_total": "{:.2f}", "p_over": "{:.2f}", "log_loss": "{:.2f}"})
    .hide(axis="index")
)

## Les matchs aussi prédits en live

`traced_live` marque les matchs prédits en live avant leur coup d'envoi (journal prospectif, critère E.2). Leur prédiction du rejeu vient du modèle du test scellé, celle du live du modèle actif : ce sont deux lignes de deux calculs, ici on ne lit que le rejeu.

In [ ]:
live = jeu[jeu["traced_live"]]
print(len(live), "matchs tracés en live sur", len(jeu))
if len(live):
    show(flat(rs.table(live, ["period"], RESAMPLES)))

## Pour aller plus loin

- Croiser une variable avec l'erreur : `jeu.groupby(pd.qcut(jeu["home_goals_for_ewm_h240"], 5))["log_loss"].mean()` (vide en 2022-23, voir plus haut).
- Comparer deux sous-ensembles (huis clos, début de saison, grands écarts d'Elo) par `rs.delta_interval(sous_ensemble, RESAMPLES)`.
- Retrouver exactement le rapport : `reports/inference/rejeu_stats_<date>.md` (10 000 tirages).
- Rappel : 2021-22 à 2024-25 ont servi à choisir les modèles. Une différence trouvée ici est une piste à confirmer sur des matchs vierges, pas une conclusion.